<a href="https://colab.research.google.com/github/Kidzantso/HireGenie/blob/main/Agent5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain langchain-community langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
get_ipython().system('sudo apt-get update && sudo apt-get install -y zstd')

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,607 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securit

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess

ollama_process = subprocess.Popen(["ollama", "serve"])

In [5]:
!ollama pull llama3

In [6]:
from typing import TypedDict, List

class InterviewState(TypedDict):
    job_description: str
    candidate_summary: str
    candidate_projects: str

    technical_questions: List[str]
    behavioral_questions: List[str]

    final_report: str

In [7]:
TECHNICAL_PROMPT = """
You are a senior technical interviewer.

Your task is to generate interview questions based on:

Job Description:
{job_description}

Candidate Summary:
{candidate_summary}

Candidate Projects:
{candidate_projects}

Generate:

1. Five job-related technical questions.
2. Three project-specific questions based on the candidate's projects.
3. Two advanced follow-up questions.

For EACH question include:

Question:
Skill Measured:

Requirements:
- Every technical question must be directly linked to a required skill from the job description.
- Clearly mention the skill being assessed.
- If the question is project-based, identify the relevant project and the skill being evaluated.
- Focus on practical understanding rather than theoretical definitions.
- Questions should become progressively more challenging.
- Avoid generic interview questions.

Output format:

Question: ...
Skill Measured: ...

Question: ...
Skill Measured: ...
"""
BEHAVIORAL_PROMPT = """
You are an experienced HR interviewer.

Your task is to generate situational and behavioral interview questions.

Job Description:
{job_description}

Candidate Summary:
{candidate_summary}

Generate 5 situational/behavioral questions.

For EACH question include:

Question:
Soft Skill Measured:

Requirements:
- Questions should be realistic workplace scenarios.
- Each question must evaluate a specific soft skill.
- Use soft skills such as:
    - Communication
    - Leadership
    - Teamwork
    - Conflict Resolution
    - Adaptability
    - Problem Solving
    - Time Management
    - Accountability
    - Decision Making
    - Resilience

- Clearly identify which soft skill is being measured.
- Use STAR-style behavioral questioning when appropriate.
- Avoid generic questions such as:
    "Tell me about yourself"
    "What are your strengths?"

Output format:

Question: ...
Soft Skill Measured: ...

Question: ...
Soft Skill Measured: ...
"""

In [9]:
from langchain_core.prompts import ChatPromptTemplate

technical_prompt_template = ChatPromptTemplate.from_template(
    TECHNICAL_PROMPT
)

behavioral_prompt_template = ChatPromptTemplate.from_template(
    BEHAVIORAL_PROMPT
)

In [12]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3")

technical_question_chain = (
    technical_prompt_template
    | llm
)

behavioral_question_chain = (
    behavioral_prompt_template
    | llm
)

In [13]:
def generate_technical_questions(state):

    response = technical_question_chain.invoke({
        "job_description": state["job_description"],
        "candidate_summary": state["candidate_summary"],
        "candidate_projects": state["candidate_projects"]
    })

    return {"technical_questions": response.content.split("\n")}



def generate_behavioral_questions(state):
    response = behavioral_question_chain.invoke({
        "job_description": state["job_description"],
        "candidate_summary": state["candidate_summary"]
    })

    return {"behavioral_questions": response.content.split("\n")}


def compile_report(state):

    report = f"""
# AI Interview Question Report

==================================================

## Technical Questions

{chr(10).join(state["technical_questions"])}

==================================================

## Behavioral Questions

{chr(10).join(state["behavioral_questions"])}

==================================================
"""

    return {"final_report": report}

In [14]:
from langgraph.graph import StateGraph, END

builder = StateGraph(InterviewState)

builder.add_node(
    "technical_questions",
    generate_technical_questions
)

builder.add_node(
    "behavioral_questions",
    generate_behavioral_questions
)

builder.add_node(
    "compile_report",
    compile_report
)

builder.set_entry_point(
    "technical_questions"
)

builder.add_edge(
    "technical_questions",
    "behavioral_questions"
)

builder.add_edge(
    "behavioral_questions",
    "compile_report"
)

builder.add_edge(
    "compile_report",
    END
)

graph = builder.compile()

In [15]:
job_description = """
Looking for a Machine Learning Engineer with:
- Python
- NLP
- Deep Learning
- LangChain
- Vector Databases
"""
candidate_summary = """
Candidate has strong Python and TensorFlow experience.
Built NLP projects and AI chatbots.
Good teamwork and communication skills.
Limited experience with vector databases.
"""
candidate_projects = """
1. AI Chatbot using LangChain and FAISS
2. Smart Attendance System using Face Recognition
3. Resume Screening System using NLP
"""

In [16]:
result = graph.invoke({
    "job_description": job_description,
    "candidate_summary": candidate_summary,
    "candidate_projects": candidate_projects
})

print(result["final_report"])


# AI Interview Question Report


## Technical Questions

Here are five job-related technical questions, three project-specific questions, and two advanced follow-up questions:

**Job-Related Technical Questions**

1. Question: Write a Python function to tokenize text data using NLTK and perform basic preprocessing (stopword removal, stemming) for an NLP task.
Skill Measured: Python

Requirements: Implement tokenization and basic preprocessing techniques to prepare the text data for further analysis.

2. Question: Design a simple neural network in TensorFlow to classify text as positive or negative sentiment. Consider using word embeddings like Word2Vec or GloVe.
Skill Measured: Deep Learning, TensorFlow

Requirements: Create a neural network architecture that can classify text sentiment using word embeddings.

3. Question: Implement a LangChain-based conversational AI system to respond to user input (e.g., "What's the weather like today?").
Skill Measured: LangChain

Requirements: Des

In [ ]:
from IPython.display import Image

Image(graph.get_graph().draw_mermaid_png())